# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² Mental Health Survey dataset using the `mlcroissant` library, closely following best practices for reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All references use their Croissant `@id` for accuracy and compatibility with the Croissant data model.

In [ ]:
# List out all record sets and their info
record_sets = list(dataset.record_sets)
print("Available Record Sets and their @id's:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")

# For each record set, list fields and columns
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id={rs.id})")
    print("Fields and their @id's:")
    for field in rs.fields:
        print(f"  - {field.name}: @id={field.id}, data_type={field.data_type}")
    print("Columns and their @id's:")
    for col in rs.columns:
        print(f"  - {col.name}: @id={col.id}")

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis, referencing specific record set and field `@id`s found above.

In [ ]:
# You may need to adapt these @ids if the actual record set @ids differ
# We'll use the main record set that contains survey responses for analysis
# Extract all available record sets, load their records into DataFrames

# List all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set '{record_set_id}'. Columns:")
            print(df.columns.tolist())
        else:
            print(f"No records found for record set '{record_set_id}'.")
    except Exception as e:
        print(f"Error loading records from '{record_set_id}': {e}")

# For demonstration, let's pick the first available non-empty record set
primary_record_set_id = next(iter(dataframes))
df = dataframes[primary_record_set_id]
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric columns, and grouping data by key attributes. All references use the associated `@id` for fields.

Examples below assume the survey response record set contains demographic and scale scores such as `GAD-7`, `PHQ-9`, and `PSQ`.

In [ ]:
# Select a numeric field (by its @id) for demonstration.
# Update this if the actual field names/@ids differ. We'll attempt to find a GAD-7 or PHQ-9 score column.
import numpy as np

numeric_candidates = ['gad_7_total_score', 'phq_9_total_score', 'psq_total_score', 'age']
numeric_field_id = None
for candidate in numeric_candidates:
    if candidate in df.columns:
        numeric_field_id = candidate
        break

if numeric_field_id is not None:
    print(f"Using numeric field: '{numeric_field_id}'\n")

    # Select a threshold for filtering
    threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 10
    # Remove missing values for this field
    filtered_df = df[df[numeric_field_id].notnull() & (df[numeric_field_id] > threshold)]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}, count: {len(filtered_df)}\n")
    print(filtered_df.head())

    # Normalize the selected field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' (z-score):")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field, e.g., 'gender' or other categorical field
    group_field_candidates = ['gender', 'level_of_education', 'marital_status']
    group_field_id = None
    for candidate in group_field_candidates:
        if candidate in df.columns:
            group_field_id = candidate
            break
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by '{group_field_id}':")
        print(grouped_df)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the normalized score and the mean score by group.

**Note:** This requires matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    # Histogram of normalized scores
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True, color='royalblue')
    plt.title(f"Distribution of normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.ylabel("Count")
    plt.show()

    # Bar plot of means by group
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(7,4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, palette="viridis")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and visualize the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using `mlcroissant` and Pandas. By referencing all entities with their Croissant `@id`, we ensure transparency and reproducibility. This dataset enables rich analysis of mental health indicators and demographic associations for the Kilifi community.